# Fannie Mae Loan Performance Data - CSV to Parquet
For each quarter: set the CSV path and quarter label, then run all cells.
Processes in chunks (RAM-safe), saves to Parquet, done.

In [12]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

## CONFIG — Only edit this cell each quarter

In [53]:
# ---- EDIT THESE TWO LINES EACH QUARTER ----
CSV_PATH = "2022Q3.csv"   # Path to the downloaded CSV file
QUARTER  = "2022Q3"       # Label for output file e.g. '2022Q1', '2024Q4'
# -------------------------------------------

OUTPUT_DIR  = "data/fannie_mae"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{QUARTER}.parquet")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Quarter : {QUARTER}")
print(f"CSV     : {CSV_PATH}")
print(f"Output  : {OUTPUT_PATH}")

if not os.path.exists(CSV_PATH):
    print(f"WARNING: {CSV_PATH} not found — check the path")
else:
    size_gb = os.path.getsize(CSV_PATH) / 1e9
    print(f"CSV size: {size_gb:.2f} GB")

Quarter : 2022Q3
CSV     : 2022Q3.csv
Output  : data/fannie_mae/2022Q3.parquet
CSV size: 4.27 GB


## Schema + Feature Selection (do not edit)

In [54]:
field_names = [
    "Reference Pool ID", "Loan Identifier", "Monthly Reporting Period",
    "Channel", "Seller Name", "Servicer Name", "Master Servicer",
    "Original Interest Rate", "Current Interest Rate",
    "Original UPB", "UPB at Issuance", "Current Actual UPB",
    "Original Loan Term", "Origination Date", "First Payment Date",
    "Loan Age", "Remaining Months to Legal Maturity",
    "Remaining Months To Maturity", "Maturity Date",
    "Original Loan to Value Ratio (LTV)",
    "Original Combined Loan to Value Ratio (CLTV)",
    "Number of Borrowers", "Debt-To-Income (DTI)",
    "Borrower Credit Score at Origination",
    "Co-Borrower Credit Score at Origination",
    "First Time Home Buyer Indicator", "Loan Purpose",
    "Property Type", "Number of Units", "Occupancy Status",
    "Property State",
    "Metropolitan Statistical Area (MSA) or Metropolitan Statistical Division Area (MSDA)",
    "Zip Code Short", "Mortgage Insurance Percentage",
    "Amortization Type", "Prepayment Penalty Indicator",
    "Interest Only Loan Indicator",
    "Interest Only First Principal And Interest Payment Date",
    "Months to Amortization", "Current Loan Delinquency Status",
    "Loan Payment History", "Modification Flag",
    "Mortgage Insurance Cancellation Indicator",
    "Zero Balance Code", "Zero Balance Effective Date",
    "UPB at the Time of Removal", "Repurchase Date",
    "Scheduled Principal Current", "Total Principal Current",
    "Unscheduled Principal Current", "Last Paid Installment Date",
    "Foreclosure Date", "Disposition Date", "Foreclosure Costs",
    "Property Preservation and Repair Costs", "Asset Recovery Costs",
    "Miscellaneous Holding Expenses and Credits",
    "Associated Taxes for Holding Property", "Net Sales Proceeds",
    "Credit Enhancement Proceeds", "Repurchase Make Whole Proceeds",
    "Other Foreclosure Proceeds",
    "Modification-Related Non-Interest Bearing UPB",
    "Principal Forgiveness Amount", "Original List Start Date",
    "Original List Price", "Current List Start Date",
    "Current List Price", "Borrower Credit Score At Issuance",
    "Co-Borrower Credit Score At Issuance",
    "Borrower Credit Score Current", "Co-Borrower Credit Score Current",
    "Mortgage Insurance Type", "Servicing Activity Indicator",
    "Current Period Modification Loss Amount",
    "Cumulative Modification Loss Amount",
    "Current Period Credit Event Net Gain or Loss",
    "Cumulative Credit Event Net Gain or Loss",
    "Special Eligibility Program", "Foreclosure Principal Write-off Amount",
    "Relocation Mortgage Indicator", "Zero Balance Code Change Date",
    "Loan Holdback Indicator", "Loan Holdback Effective Date",
    "Delinquent Accrued Interest", "Property Valuation Method",
    "High Balance Loan Indicator",
    "ARM Initial Fixed-Rate Period <= 5 YR Indicator",
    "ARM Product Type", "Initial Fixed-Rate Period",
    "Interest Rate Adjustment Frequency",
    "Next Interest Rate Adjustment Date", "Next Payment Change Date",
    "Index", "ARM Cap Structure",
    "Initial Interest Rate Cap Up Percent",
    "Periodic Interest Rate Cap Up Percent",
    "Lifetime Interest Rate Cap Up Percent", "Mortgage Margin",
    "ARM Balloon Indicator", "ARM Plan Number",
    "Borrower Assistance Plan",
    "High Loan to Value (HLTV) Refinance Option Indicator",
    "Deal Name", "Repurchase Make Whole Proceeds Flag",
    "Alternative Delinquency Resolution",
    "Alternative Delinquency Resolution Count",
    "Total Deferral Amount",
    "Payment Deferral Modification Event Indicator",
    "Interest Bearing UPB",
]

KEEP_COLS = [
    # Identifiers
    "Loan Identifier",
    "Monthly Reporting Period",
    # Static origination features
    "Origination Date",
    "Channel",
    "Original Interest Rate",
    "Original UPB",
    "Original Loan Term",
    "Original Loan to Value Ratio (LTV)",
    "Original Combined Loan to Value Ratio (CLTV)",
    "Debt-To-Income (DTI)",
    "Borrower Credit Score at Origination",
    "Co-Borrower Credit Score at Origination",
    "Number of Borrowers",
    "First Time Home Buyer Indicator",
    "Loan Purpose",
    "Property Type",
    "Occupancy Status",
    "Property State",
    "Mortgage Insurance Percentage",
    # Dynamic performance features
    "Current Interest Rate",
    "Current Actual UPB",
    "Loan Age",
    "Remaining Months to Legal Maturity",
    "Remaining Months To Maturity",
    "Current Loan Delinquency Status",
    "Loan Payment History",
    "Total Principal Current",
    "Modification Flag",
    "Borrower Assistance Plan",
    "Servicing Activity Indicator",
]

print(f"Keeping {len(KEEP_COLS)} of {len(field_names)} columns")

Keeping 30 of 110 columns


## Step 1 — Process CSV → Parquet (chunked)

In [55]:
if os.path.exists(OUTPUT_PATH):
    print(f"{OUTPUT_PATH} already exists — delete it first if you want to reprocess")
else:
    CHUNK_SIZE = 500_000
    writer     = None
    total_rows = 0
    chunk_num  = 0

    print(f"Processing {QUARTER} in {CHUNK_SIZE:,}-row chunks...")

    for chunk in pd.read_csv(
        CSV_PATH,
        sep="|",
        header=None,
        names=field_names,
        chunksize=CHUNK_SIZE,
        engine="python",
        on_bad_lines="skip",
    ):
        chunk_num += 1

        # Keep only selected columns
        available = [c for c in KEEP_COLS if c in chunk.columns]
        chunk = chunk[available].copy()

        # Drop fully empty rows
        chunk = chunk.dropna(how="all")

        # Force consistent dtypes across chunks — pandas infers int64 when no nulls,
        # float64 when nulls present. Normalize to float64 to keep Parquet schema stable.
        float_cols = [
            "Loan Age",
            "Remaining Months to Legal Maturity",
            "Remaining Months To Maturity",
            "Original Loan Term",
            "Original Loan to Value Ratio (LTV)",
            "Original Combined Loan to Value Ratio (CLTV)",
            "Number of Borrowers",
            "Debt-To-Income (DTI)",
        ]
        for col in float_cols:
            if col in chunk.columns:
                chunk[col] = chunk[col].astype("float64")

        # Parse date columns from Fannie Mae MMYYYY format
        for col in ["Monthly Reporting Period", "Origination Date"]:
            if col in chunk.columns:
                chunk[col] = pd.to_datetime(
                    chunk[col].astype(str), format="%m%Y", errors="coerce"
                )

        # Write chunk to Parquet incrementally
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PATH, table.schema, compression="snappy")
        writer.write_table(table)

        total_rows += len(chunk)
        print(f"  Chunk {chunk_num} — {total_rows:,} rows written", end="\r")

    if writer:
        writer.close()

    print(f"\nDone. {total_rows:,} rows → {OUTPUT_PATH}")

Processing 2022Q3 in 500,000-row chunks...
  Chunk 27 — 13,375,587 rows written
Done. 13,375,587 rows → data/fannie_mae/2022Q3.parquet


## Step 2 — Summary of all quarters collected

In [56]:
saved = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".parquet")])
print(f"Quarters collected ({len(saved)}):")
for fname in saved:
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:<20} {size_mb:>8.0f} MB")

Quarters collected (15):
  2022Q1.parquet            395 MB
  2022Q2.parquet            252 MB
  2022Q3.parquet            150 MB
  2022Q4.parquet             99 MB
  2023Q1.parquet             70 MB
  2023Q2.parquet             84 MB
  2023Q3.parquet             73 MB
  2023Q4.parquet             51 MB
  2024Q1.parquet             40 MB
  2024Q2.parquet             46 MB
  2024Q3.parquet             42 MB
  2024Q4.parquet             30 MB
  2025Q1.parquet             18 MB
  2025Q2.parquet             17 MB
  2025Q3.parquet             10 MB


## Step 3 — Verify

In [57]:
df = pd.read_parquet(OUTPUT_PATH)

print(f"Shape          : {df.shape}")
print(f"Unique loans   : {df['Loan Identifier'].nunique():,}")
print(f"Date range     : {df['Monthly Reporting Period'].min()} → {df['Monthly Reporting Period'].max()}")
print(f"\nDelinquency distribution (top 10):")
print(df["Current Loan Delinquency Status"].value_counts().head(10))

df.head(3)

Shape          : (13375587, 30)
Unique loans   : 377,913
Date range     : 2022-07-01 00:00:00 → 2025-09-01 00:00:00

Delinquency distribution (top 10):
Current Loan Delinquency Status
0    13109252
1      140815
2       38206
3       18975
4       14160
5       11969
6        8265
7        6428
8        5395
9        4390
Name: count, dtype: int64


,Loan Identifier,Monthly Reporting Period,Origination Date,Channel,Original Interest Rate,Original UPB,Original Loan Term,Original Loan to Value Ratio (LTV),Original Combined Loan to Value Ratio (CLTV),Debt-To-Income (DTI),...,Current Actual UPB,Loan Age,Remaining Months to Legal Maturity,Remaining Months To Maturity,Current Loan Delinquency Status,Loan Payment History,Total Principal Current,Modification Flag,Borrower Assistance Plan,Servicing Activity Indicator
0,133801946,2022-07-01,2022-06-01,R,5.49,140000.0,360.0,53.0,53.0,45.0,...,140000.0,0.0,360.0,360.0,0,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX00,NaN,N,7,N
1,133801946,2022-08-01,2022-06-01,R,5.49,140000.0,360.0,53.0,53.0,45.0,...,140000.0,1.0,359.0,359.0,0,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX0000,153.53,N,7,N
2,133801946,2022-09-01,2022-06-01,R,5.49,140000.0,360.0,53.0,53.0,45.0,...,140000.0,2.0,358.0,359.0,1,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX000001,0.00,N,N,N
